In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms
import timm

In [2]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [3]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "mobile-vision-transformer",
    "learning_rate": 0.001,
    "epochs": 20,
    "pretrained":True
}

In [4]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ])
}

train_dir = "../BanglaLekha_dataset_split/train"
val_dir = "../BanglaLekha_dataset_split/validation"
test_dir = "../BanglaLekha_dataset_split/test"


train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])
test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms["test"])

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [5]:

mobilevit_s = timm.create_model("mobilevit_s", pretrained=True)

for params in mobilevit_s.parameters():
    params.requires_grad = False

mobilevit_s.reset_classifier(84)


total_params = sum(p.numel() for p in mobilevit_s.parameters())

gpu = torch.device("cuda")
mobilevit_s = mobilevit_s.to(gpu)


In [6]:
print(mobilevit_s)

ByobNet(
  (stem): ConvNormAct(
    (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
  )
  (stages): Sequential(
    (0): Sequential(
      (0): BottleneckBlock(
        (conv1_1x1): ConvNormAct(
          (conv): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNormAct2d(
            64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
        )
        (conv2_kxk): ConvNormAct(
          (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
          (bn): BatchNormAct2d(
            64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
   

In [7]:
print(total_params)

4991476


In [15]:
import fine_tuning as ft

mobilevit_s = ft.fine_tune(model=mobilevit_s, model_name="mobilevit_s", state="full") #Change the state for fine tuning 

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW([
    {"params": mobilevit_s.head.parameters(), "lr": 1e-3, "weight_decay": 1e-4},
    {"params": mobilevit_s.stages.parameters(), "lr": 1e-5, "weight_decay": 1e-4},
    
])
epochs = model_config["epochs"]

In [10]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [11]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs):
    
    max_val_accuracy = 0
    patience = 5
    count = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
        
        val_accuracy = validate_model(mobilevit_s, val_dataloader)

        if(val_accuracy > max_val_accuracy):
            max_val_accuracy = val_accuracy
            count = 0

            torch.save(model.state_dict(), "saved_parameters/mobile_ViT.pth")
        else:
            count = count + 1

        if(count >= patience):
            break
        

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}, Validation Accuracy: {val_accuracy}")

In [17]:
train_model(mobilevit_s, train_dataloader, val_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 0.1896981216717886, Validation Accuracy: 0.9121780565775982
Epoch: 2, Training Loss: 0.16771717076585838, Validation Accuracy: 0.9158574099764762
Epoch: 3, Training Loss: 0.15187590072281335, Validation Accuracy: 0.9179081971168346
Epoch: 4, Training Loss: 0.1379780448840757, Validation Accuracy: 0.9022257072199771
Epoch: 5, Training Loss: 0.1248024124724608, Validation Accuracy: 0.9170637553531577
Epoch: 6, Training Loss: 0.11321679061636365, Validation Accuracy: 0.9166415344713191
Epoch: 7, Training Loss: 0.10166682687398984, Validation Accuracy: 0.9184510525363412
Epoch: 8, Training Loss: 0.09251206307310697, Validation Accuracy: 0.9174859762349961
Epoch: 9, Training Loss: 0.08434700798167237, Validation Accuracy: 0.9165812172024851
Epoch: 10, Training Loss: 0.07690943264992593, Validation Accuracy: 0.9152542372881356
Epoch: 11, Training Loss: 0.06948679793095872, Validation Accuracy: 0.9170637553531577


In [18]:
mobilevit_s.load_state_dict(torch.load("saved_parameters/mobile_ViT.pth"))
mobilevit_s.eval()

ByobNet(
  (stem): ConvNormAct(
    (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn): BatchNormAct2d(
      16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
  )
  (stages): Sequential(
    (0): Sequential(
      (0): BottleneckBlock(
        (conv1_1x1): ConvNormAct(
          (conv): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNormAct2d(
            64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
        )
        (conv2_kxk): ConvNormAct(
          (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=64, bias=False)
          (bn): BatchNormAct2d(
            64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
   

In [19]:
accuracy = validate_model(mobilevit_s, test_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 91.57679714019646
